In [ ]:
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import pingouin as pg


In [ ]:
# Bayesian comparison of semantic distance between probes and targets vs probes and distractors
# Step 1: Read the Excel file
file_path = "/SJTstim-info.xlsx" #change this to the path where you have the SJTstim-info.xlsx file saved
semantic_df = pd.read_excel(file_path, sheet_name="Semantic Distances")

# Step 2: Keep only the distance columns we need and clean them
semantic_df = semantic_df[["Probe", "PTSD", "PD1SD", "PD2SD"]].copy()
for col in ["PTSD", "PD1SD", "PD2SD"]:
    semantic_df[col] = pd.to_numeric(semantic_df[col], errors="coerce")

semantic_df = semantic_df.dropna(subset=["PTSD", "PD1SD", "PD2SD"]).reset_index(drop=True)
semantic_df["DistractorMean"] = semantic_df[["PD1SD", "PD2SD"]].mean(axis=1)

# Step 3: Reshape to long format for later plotting and inspection
df_long = pd.melt(
    semantic_df,
    id_vars=["Probe"],
    value_vars=["PTSD", "PD1SD", "PD2SD"],
    var_name="PairType",
    value_name="SemanticDistance",
)

df_long["PairType"] = df_long["PairType"].map(
    {
        "PTSD": "Target-Probe",
        "PD1SD": "Distractor 1-Probe",
        "PD2SD": "Distractor 2-Probe",
    }
)

print(df_long.head())

# Step 4: Run Bayesian paired t-tests and extract Bayes factors
comparisons = [
    ("PTSD", "DistractorMean", "Target-Probe vs Mean Distractor-Probe"),
    ("PTSD", "PD1SD", "Target-Probe vs Distractor 1-Probe"),
    ("PTSD", "PD2SD", "Target-Probe vs Distractor 2-Probe"),
]

def bf10_to_float(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()
    if text.lower() in {"inf", "infinity"}:
        return np.inf
    if text.startswith(">"):
        return float(text[1:]) if text[1:].replace(".", "", 1).isdigit() else np.inf
    if text.startswith("<"):
        return float(text[1:]) if text[1:].replace(".", "", 1).isdigit() else 0.0
    return pd.to_numeric(text, errors="coerce")

bayes_rows = []

for left_col, right_col, label in comparisons:
    paired = semantic_df[[left_col, right_col]].dropna()
    test_result = pg.ttest(paired[left_col], paired[right_col], paired=True)
    test_row = test_result.iloc[0]
    mean_diff = (paired[left_col] - paired[right_col]).mean()
    bf10_value = bf10_to_float(test_row["BF10"])

    bayes_rows.append(
        {
            "Comparison": label,
            "N": len(paired),
            "MeanDifference": mean_diff,
            "t-statistic": test_row["T"],
            "p-value": test_row["p-val"],
            "BayesFactor10": bf10_value,
            "BayesFactor10Raw": test_row["BF10"],
            "Cohens_d": test_row["cohen-d"],
        }
    )

pairwise_df = pd.DataFrame(bayes_rows)

# Step 5: Print results 
def interpret_bayes_factor(bf10):
    if pd.isna(bf10):
        return "Bayes factor unavailable"
    if bf10 < 1 / 3:
        return "evidence for the null"
    if bf10 < 1:
        return "anecdotal evidence for the null"
    if bf10 < 3:
        return "anecdotal evidence for the alternative"
    if bf10 < 10:
        return "moderate evidence for the alternative"
    return "strong evidence for the alternative"

pairwise_df["Evidence"] = pairwise_df["BayesFactor10"].apply(interpret_bayes_factor)
print(pairwise_df)


        Probe      PairType  SemanticDistance
0       YODEL  Target-Probe          1.135022
1     INERTIA  Target-Probe          1.002449
2  GYMNASTICS  Target-Probe          0.995840
3     INTERIM  Target-Probe          0.983059
4    OUTGOING  Target-Probe          0.971086
                              Comparison    N  MeanDifference  t-statistic  \
0  Target-Probe vs Mean Distractor-Probe  449       -0.224943   -26.085150   
1     Target-Probe vs Distractor 1-Probe  449       -0.226689   -23.843158   
2     Target-Probe vs Distractor 2-Probe  449       -0.223196   -25.277837   

        p-value  BayesFactor10 BayesFactor10Raw  Cohens_d  \
0  6.558331e-92   1.553000e+88        1.553e+88  1.540749   
1  9.910125e-82   1.156000e+78        1.156e+78  1.479886   
2  2.913267e-88   3.643000e+84        3.643e+84  1.453733   

                              Evidence  
0  strong evidence for the alternative  
1  strong evidence for the alternative  
2  strong evidence for the alternative  
